In [22]:
%cd /content

!git clone https://github.com/ultralytics/yolov5.git
%cd /content/yolov5

!pip install -qr requirements.txt
!pip install thop pyyaml


/content
fatal: destination path 'yolov5' already exists and is not an empty directory.
/content/yolov5


In [23]:
!nvidia-smi


Sat Mar 14 05:44:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   29C    P0             48W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [48]:
# Cell 4 — REPLACE ENTIRE CELL WITH THIS
from pathlib import Path
import os
import subprocess
import re

ROOT = Path("/content/yolov5")
WORK_DIR = Path("/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study")
MODEL_DIR = WORK_DIR / "models_custom"
DATA_DIR = WORK_DIR / "data_custom"
RUNS_DIR = WORK_DIR / "runs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ROOT = "/content/drive/MyDrive/Lab thầy Vinh/Train_VOC_Test_Dawn/Train_VOC_Test_Dawn"

print("ROOT:", ROOT)
print("WORK_DIR:", WORK_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("DATA_DIR:", DATA_DIR)
print("RUNS_DIR:", RUNS_DIR)
print("DATASET_ROOT:", DATASET_ROOT)

def run_cmd(cmd, cwd="/content/yolov5"):
    """
    Stream subprocess output live to Colab and also return the full text.
    """
    print("\n" + "=" * 100)
    print("RUNNING:")
    print(" ".join(str(x) for x in cmd))
    print("=" * 100)

    process = subprocess.Popen(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    collected = []
    for line in iter(process.stdout.readline, ''):
        if line:
            print(line, end="")
            collected.append(line)

    process.stdout.close()
    return_code = process.wait()
    output = "".join(collected)

    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd, output=output)

    return output

def extract_test_all_metrics(output_text):
    """
    Try to parse the final 'all' row from YOLOv5 val.py output.
    Expected shape:
    all   1015   7200   0.505   0.230   0.257   0.146
    """
    lines = output_text.splitlines()
    candidates = []

    for line in lines:
        s = line.strip()
        if s.startswith("all"):
            parts = s.split()
            if len(parts) >= 7:
                candidates.append(parts)

    if not candidates:
        return None

    parts = candidates[-1]
    try:
        return {
            "images": int(parts[1]),
            "instances": int(parts[2]),
            "P": float(parts[3]),
            "R": float(parts[4]),
            "mAP50": float(parts[5]),
            "mAP50-95": float(parts[6]),
        }
    except:
        return None


ROOT: /content/yolov5
WORK_DIR: /content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study
MODEL_DIR: /content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/models_custom
DATA_DIR: /content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/data_custom
RUNS_DIR: /content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/runs
DATASET_ROOT: /content/drive/MyDrive/Lab thầy Vinh/Train_VOC_Test_Dawn/Train_VOC_Test_Dawn


In [25]:
data_yaml = f"""path: {DATASET_ROOT}
train: train/images
val: val/images
test: test/images

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: bus
  5: train
"""

data_yaml_path = DATA_DIR / "train_voc_test_dawn.yaml"
data_yaml_path.write_text(data_yaml, encoding="utf-8")

print(data_yaml_path)
print(data_yaml)


/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/data_custom/train_voc_test_dawn.yaml
path: /content/drive/MyDrive/Lab thầy Vinh/Train_VOC_Test_Dawn/Train_VOC_Test_Dawn
train: train/images
val: val/images
test: test/images

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: bus
  5: train



In [26]:
hyp_yaml = """
lr0: 0.01
lrf: 0.01
momentum: 0.937
weight_decay: 0.0005
warmup_epochs: 3.0
warmup_momentum: 0.8
warmup_bias_lr: 0.1

box: 0.05
cls: 0.5
cls_pw: 1.0
obj: 1.0
obj_pw: 1.0
iou_t: 0.20
anchor_t: 4.0
fl_gamma: 0.0

hsv_h: 0.015
hsv_s: 0.7
hsv_v: 0.4
degrees: 0.0
translate: 0.1
scale: 0.5
shear: 0.0
perspective: 0.0
flipud: 0.0
fliplr: 0.5
mosaic: 0.0
mixup: 0.0
copy_paste: 0.0
"""

hyp_yaml_path = DATA_DIR / "hyp_rescbam4_attention.yaml"
hyp_yaml_path.write_text(hyp_yaml, encoding="utf-8")

print(hyp_yaml_path)
print(hyp_yaml)


/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/data_custom/hyp_rescbam4_attention.yaml

lr0: 0.01
lrf: 0.01
momentum: 0.937
weight_decay: 0.0005
warmup_epochs: 3.0
warmup_momentum: 0.8
warmup_bias_lr: 0.1

box: 0.05
cls: 0.5
cls_pw: 1.0
obj: 1.0
obj_pw: 1.0
iou_t: 0.20
anchor_t: 4.0
fl_gamma: 0.0

hsv_h: 0.015
hsv_s: 0.7
hsv_v: 0.4
degrees: 0.0
translate: 0.1
scale: 0.5
shear: 0.0
perspective: 0.0
flipud: 0.0
fliplr: 0.5
mosaic: 0.0
mixup: 0.0
copy_paste: 0.0



In [58]:
from pathlib import Path

custom_attention_code = r'''
import torch
import torch.nn as nn
import torch.nn.functional as F


class BasicConv(nn.Module):
    def __init__(self, in_planes, out_planes, kernel_size, stride=1, padding=0, groups=1, relu=True, bn=True, bias=False):
        super().__init__()
        self.conv = nn.Conv2d(
            in_planes, out_planes, kernel_size=kernel_size, stride=stride,
            padding=padding, dilation=1, groups=groups, bias=bias
        )
        self.bn = nn.BatchNorm2d(out_planes) if bn else nn.Identity()
        self.relu = nn.ReLU(inplace=True) if relu else nn.Identity()

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))


class Flatten(nn.Module):
    def forward(self, x):
        return x.view(x.size(0), -1)


class ChannelGate(nn.Module):
    def __init__(self, gate_channels, reduction_ratio=16):
        super().__init__()
        mid = max(8, gate_channels // reduction_ratio)
        self.mlp = nn.Sequential(
            Flatten(),
            nn.Linear(gate_channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, gate_channels)
        )

    def forward(self, x):
        avg_pool = F.adaptive_avg_pool2d(x, 1)
        max_pool = F.adaptive_max_pool2d(x, 1)

        avg_att = self.mlp(avg_pool)
        max_att = self.mlp(max_pool)

        scale = torch.sigmoid(avg_att + max_att).unsqueeze(2).unsqueeze(3)
        return x * scale.expand_as(x)


class SpatialGate(nn.Module):
    def __init__(self):
        super().__init__()
        self.spatial = BasicConv(2, 1, kernel_size=7, stride=1, padding=3, relu=False)

    def forward(self, x):
        x_compress = torch.cat(
            (torch.max(x, 1)[0].unsqueeze(1), torch.mean(x, 1).unsqueeze(1)),
            dim=1
        )
        x_out = self.spatial(x_compress)
        scale = torch.sigmoid(x_out)
        return x * scale


class CBAMBlock(nn.Module):
    def __init__(self, channels, reduction_ratio=16):
        super().__init__()
        self.channel_gate = ChannelGate(channels, reduction_ratio)
        self.spatial_gate = SpatialGate()

    def forward(self, x):
        x = self.channel_gate(x)
        x = self.spatial_gate(x)
        return x


class ResCBAM(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.cv1 = nn.Conv2d(channels, channels, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.cv2 = nn.Conv2d(channels, channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.act = nn.SiLU(inplace=True)
        self.attn = CBAMBlock(channels)

    def _get_act(self):
        if hasattr(self, "act"):
            return self.act
        if hasattr(self, "relu"):
            return self.relu
        if hasattr(self, "leakyrelu"):
            return self.leakyrelu
        return nn.SiLU(inplace=True)

    def _get_attn(self):
        if hasattr(self, "attn"):
            return self.attn
        if hasattr(self, "cbam"):
            return self.cbam
        if hasattr(self, "attention"):
            return self.attention
        return nn.Identity()

    def forward(self, x):
        identity = x
        act = self._get_act()
        attn = self._get_attn()

        out = act(self.bn1(self.cv1(x)))
        out = self.bn2(self.cv2(out))
        out = attn(out)
        out = act(out + identity)
        return out


class SimAM(nn.Module):
    def __init__(self, channels=None, e_lambda=1e-4):
        super().__init__()
        self.activaton = nn.Sigmoid()
        self.e_lambda = e_lambda

    def forward(self, x):
        b, c, h, w = x.size()
        n = h * w - 1
        if n <= 0:
            return x
        x_minus_mu_square = (x - x.mean(dim=[2, 3], keepdim=True)).pow(2)
        y = x_minus_mu_square / (
            4 * (x_minus_mu_square.sum(dim=[2, 3], keepdim=True) / n + self.e_lambda)
        ) + 0.5
        return x * self.activaton(y)


class h_sigmoid(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.relu = nn.ReLU6(inplace=inplace)

    def forward(self, x):
        return self.relu(x + 3) / 6


class h_swish(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.sigmoid = h_sigmoid(inplace=inplace)

    def forward(self, x):
        return x * self.sigmoid(x)


class CoordAtt(nn.Module):
    def __init__(self, inp, reduction=32):
        super().__init__()
        mip = max(8, inp // reduction)

        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))

        self.conv1 = nn.Conv2d(inp, mip, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = h_swish()

        self.conv_h = nn.Conv2d(mip, inp, kernel_size=1, stride=1, padding=0, bias=False)
        self.conv_w = nn.Conv2d(mip, inp, kernel_size=1, stride=1, padding=0, bias=False)

    def forward(self, x):
        identity = x
        n, c, h, w = x.size()

        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)

        y = torch.cat([x_h, x_w], dim=2)
        y = self.act(self.bn1(self.conv1(y)))

        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)

        a_h = torch.sigmoid(self.conv_h(x_h))
        a_w = torch.sigmoid(self.conv_w(x_w))

        return identity * a_h * a_w


class ZPool(nn.Module):
    def forward(self, x):
        return torch.cat((torch.max(x, 1)[0].unsqueeze(1), torch.mean(x, 1).unsqueeze(1)), dim=1)


class AttentionGate(nn.Module):
    def __init__(self):
        super().__init__()
        self.zpool = ZPool()
        self.conv = BasicConv(2, 1, kernel_size=7, stride=1, padding=3, relu=False)

    def forward(self, x):
        x_out = self.conv(self.zpool(x))
        scale = torch.sigmoid(x_out)
        return x * scale


class TripletAttention(nn.Module):
    def __init__(self, channels=None, no_spatial=False):
        super().__init__()
        self.cw = AttentionGate()
        self.hc = AttentionGate()
        self.no_spatial = no_spatial
        if not self.no_spatial:
            self.hw = AttentionGate()

    def forward(self, x):
        x_perm1 = x.permute(0, 2, 1, 3).contiguous()
        x_out1 = self.cw(x_perm1)
        x_out11 = x_out1.permute(0, 2, 1, 3).contiguous()

        x_perm2 = x.permute(0, 3, 2, 1).contiguous()
        x_out2 = self.hc(x_perm2)
        x_out21 = x_out2.permute(0, 3, 2, 1).contiguous()

        if not self.no_spatial:
            x_out = self.hw(x)
            return (x_out + x_out11 + x_out21) / 3.0
        return (x_out11 + x_out21) / 2.0


class NAM(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.bn = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x_bn = self.bn(x)
        weight = self.bn.weight.abs()
        weight = weight / (weight.sum() + 1e-6)
        weight = weight.view(1, -1, 1, 1)
        y = torch.sigmoid(x_bn * weight)
        return residual * y


class GCBlock(nn.Module):
    def __init__(self, inplanes, ratio=0.25):
        super().__init__()
        planes = max(1, int(inplanes * ratio))
        self.conv_mask = nn.Conv2d(inplanes, 1, kernel_size=1)
        self.softmax = nn.Softmax(dim=2)
        self.channel_add_conv = nn.Sequential(
            nn.Conv2d(inplanes, planes, kernel_size=1),
            nn.LayerNorm([planes, 1, 1]),
            nn.ReLU(inplace=True),
            nn.Conv2d(planes, inplanes, kernel_size=1)
        )

    def spatial_pool(self, x):
        batch, channel, height, width = x.size()
        input_x = x.view(batch, channel, height * width).unsqueeze(1)
        context_mask = self.conv_mask(x).view(batch, 1, height * width)
        context_mask = self.softmax(context_mask).unsqueeze(-1)
        context = torch.matmul(input_x, context_mask).view(batch, channel, 1, 1)
        return context

    def forward(self, x):
        context = self.spatial_pool(x)
        out = x + self.channel_add_conv(context)
        return out
'''

Path("/content/yolov5/models/custom_attention.py").write_text(custom_attention_code, encoding="utf-8")
print("Rewrote /content/yolov5/models/custom_attention.py with compatibility-friendly ResCBAM")

Rewrote /content/yolov5/models/custom_attention.py with compatibility-friendly ResCBAM


In [28]:
from pathlib import Path

common_path = Path("/content/yolov5/models/common.py")
text = common_path.read_text(encoding="utf-8")

import_line = "from models.custom_attention import *\n"

if import_line not in text:
    lines = text.splitlines(keepends=True)
    insert_at = 0
    for i, line in enumerate(lines):
        if line.startswith("from utils.") or line.startswith("from models.experimental"):
            insert_at = i + 1
    lines.insert(insert_at, import_line)
    text = "".join(lines)
    common_path.write_text(text, encoding="utf-8")
    print("Patched common.py")
else:
    print("common.py already patched")


common.py already patched


In [37]:
# Cell 9 — REPLACE ENTIRE CELL WITH THIS
from pathlib import Path
import re

yolo_path = Path("/content/yolov5/models/yolo.py")
text = yolo_path.read_text(encoding="utf-8")

import_line = "from models.custom_attention import *\n"

# Remove any old copies first
text = text.replace(import_line, "")

# Safe insertion strategy:
# insert AFTER the full multi-line "from models.common import (...)" block
pattern = r"(from models\.common import\s*\([\s\S]*?\)\n)"
match = re.search(pattern, text)

if match:
    insert_pos = match.end()
    text = text[:insert_pos] + import_line + text[insert_pos:]
    yolo_path.write_text(text, encoding="utf-8")
    print("Patched yolo.py right after the full models.common import block")
else:
    raise RuntimeError("Could not find the complete 'from models.common import (...)' block in models/yolo.py")


Patched yolo.py right after the full models.common import block


In [38]:
# NEW CELL — put this right after Cell 9
from pathlib import Path

yolo_path = Path("/content/yolov5/models/yolo.py")
lines = yolo_path.read_text(encoding="utf-8").splitlines()

for i, line in enumerate(lines[:90], start=1):
    print(f"{i:02d}: {line}")


01: # Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license
02: """
03: YOLO-specific modules.
04: 
05: Usage:
06:     $ python models/yolo.py --cfg yolov5s.yaml
07: """
08: 
09: import argparse
10: import contextlib
11: import math
12: import os
13: import platform
14: import sys
15: from copy import deepcopy
16: from pathlib import Path
17: 
18: import torch
19: import torch.nn as nn
20: 
21: FILE = Path(__file__).resolve()
22: ROOT = FILE.parents[1]  # YOLOv5 root directory
23: if str(ROOT) not in sys.path:
24:     sys.path.append(str(ROOT))  # add ROOT to PATH
25: if platform.system() != "Windows":
26:     ROOT = Path(os.path.relpath(ROOT, Path.cwd()))  # relative
27: 
28: from models.common import (
29:     C3,
30:     C3SPP,
31:     C3TR,
32:     SPP,
33:     SPPF,
34:     Bottleneck,
35:     BottleneckCSP,
36:     C3Ghost,
37:     C3x,
38:     Classify,
39:     Concat,
40:     Contract,
41:     Conv,
42:     CrossConv,
43:     DetectMultiBackend,
44:     DWConv,
45:  

In [39]:
from pathlib import Path

common_py = Path("/content/yolov5/models/common.py").read_text(encoding="utf-8", errors="ignore")
custom_py = Path("/content/yolov5/models/custom_attention.py").read_text(encoding="utf-8", errors="ignore")
yolo_py = Path("/content/yolov5/models/yolo.py").read_text(encoding="utf-8", errors="ignore")

needed = ["ResCBAM", "SimAM", "CoordAtt", "TripletAttention", "NAM", "GCBlock"]

print("Checking for custom module names...\n")
for m in needed:
    found = (m in common_py) or (m in custom_py) or (m in yolo_py)
    print(f"{m:18s} -> {'FOUND' if found else 'NOT FOUND'}")


Checking for custom module names...

ResCBAM            -> FOUND
SimAM              -> FOUND
CoordAtt           -> FOUND
TripletAttention   -> FOUND
NAM                -> FOUND
GCBlock            -> FOUND


In [45]:
# NEW CELL — put this right after Cell 10
from pathlib import Path
import re

general_path = Path("/content/yolov5/utils/general.py")
text = general_path.read_text(encoding="utf-8")

original = "torch.use_deterministic_algorithms(True)"
replacement = "torch.use_deterministic_algorithms(True, warn_only=True)"

if original in text:
    text = text.replace(original, replacement)
    general_path.write_text(text, encoding="utf-8")
    print("Patched utils/general.py to use warn_only=True for deterministic algorithms.")
else:
    print("Exact deterministic line not found. Showing nearby matches:")
    for m in re.finditer(r"torch\.use_deterministic_algorithms\([^\)]*\)", text):
        print("FOUND:", m.group(0))


Patched utils/general.py to use warn_only=True for deterministic algorithms.


In [46]:
# NEW CELL — put this right after the patch cell above
from pathlib import Path
import re

general_path = Path("/content/yolov5/utils/general.py")
text = general_path.read_text(encoding="utf-8")

matches = re.findall(r"torch\.use_deterministic_algorithms\([^\)]*\)", text)
print(matches)


['torch.use_deterministic_algorithms(True, warn_only=True)']


In [41]:
# Cell 11 — REPLACE ENTIRE CELL WITH THIS
model_yamls = {
    "yolov5n_rescbam4_baseline.yaml": """# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license
nc: 6
depth_multiple: 0.33
width_multiple: 0.25
anchors:
  - [10, 13, 16, 30, 33, 23]
  - [30, 61, 62, 45, 59, 119]
  - [116, 90, 156, 198, 373, 326]

backbone:
  [
    [-1, 1, Conv, [64, 6, 2, 2]],
    [-1, 1, Conv, [128, 3, 2]],
    [-1, 3, C3, [128]],
    [-1, 1, Conv, [256, 3, 2]],
    [-1, 6, C3, [256]],
    [-1, 1, Conv, [512, 3, 2]],
    [-1, 9, C3, [512]],
    [-1, 1, Conv, [1024, 3, 2]],
    [-1, 3, C3, [1024]],
    [-1, 1, SPPF, [1024, 5]],
  ]

head: [
    [-1, 1, Conv, [512, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 6], 1, Concat, [1]],
    [-1, 3, C3, [512, False]],
    [-1, 1, ResCBAM, [128]],

    [-1, 1, Conv, [256, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 4], 1, Concat, [1]],
    [-1, 3, C3, [256, False]],
    [-1, 1, ResCBAM, [64]],

    [-1, 1, Conv, [256, 3, 2]],
    [[-1, 14], 1, Concat, [1]],
    [-1, 3, C3, [512, False]],
    [-1, 1, ResCBAM, [128]],

    [-1, 1, Conv, [512, 3, 2]],
    [[-1, 10], 1, Concat, [1]],
    [-1, 3, C3, [1024, False]],
    [-1, 1, ResCBAM, [256]],

    [[19, 23, 27], 1, Detect, [nc, anchors]],
  ]
""",

    "yolov5n_rescbam4_simam.yaml": """# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license
nc: 6
depth_multiple: 0.33
width_multiple: 0.25
anchors:
  - [10, 13, 16, 30, 33, 23]
  - [30, 61, 62, 45, 59, 119]
  - [116, 90, 156, 198, 373, 326]

backbone:
  [
    [-1, 1, Conv, [64, 6, 2, 2]],
    [-1, 1, Conv, [128, 3, 2]],
    [-1, 3, C3, [128]],
    [-1, 1, Conv, [256, 3, 2]],
    [-1, 6, C3, [256]],
    [-1, 1, Conv, [512, 3, 2]],
    [-1, 9, C3, [512]],
    [-1, 1, Conv, [1024, 3, 2]],
    [-1, 3, C3, [1024]],
    [-1, 1, SPPF, [1024, 5]],
  ]

head: [
    [-1, 1, Conv, [512, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 6], 1, Concat, [1]],
    [-1, 3, C3, [512, False]],
    [-1, 1, ResCBAM, [128]],

    [-1, 1, Conv, [256, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 4], 1, Concat, [1]],
    [-1, 3, C3, [256, False]],
    [-1, 1, ResCBAM, [64]],

    [-1, 1, Conv, [256, 3, 2]],
    [[-1, 14], 1, Concat, [1]],
    [-1, 3, C3, [512, False]],
    [-1, 1, ResCBAM, [128]],

    [-1, 1, Conv, [512, 3, 2]],
    [[-1, 10], 1, Concat, [1]],
    [-1, 3, C3, [1024, False]],
    [-1, 1, ResCBAM, [256]],

    [-1, 1, SimAM, [256]],
    [[19, 23, 28], 1, Detect, [nc, anchors]],
  ]
""",

    "yolov5n_rescbam4_coordatt.yaml": """# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license
nc: 6
depth_multiple: 0.33
width_multiple: 0.25
anchors:
  - [10, 13, 16, 30, 33, 23]
  - [30, 61, 62, 45, 59, 119]
  - [116, 90, 156, 198, 373, 326]

backbone:
  [
    [-1, 1, Conv, [64, 6, 2, 2]],
    [-1, 1, Conv, [128, 3, 2]],
    [-1, 3, C3, [128]],
    [-1, 1, Conv, [256, 3, 2]],
    [-1, 6, C3, [256]],
    [-1, 1, Conv, [512, 3, 2]],
    [-1, 9, C3, [512]],
    [-1, 1, Conv, [1024, 3, 2]],
    [-1, 3, C3, [1024]],
    [-1, 1, SPPF, [1024, 5]],
  ]

head: [
    [-1, 1, Conv, [512, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 6], 1, Concat, [1]],
    [-1, 3, C3, [512, False]],
    [-1, 1, ResCBAM, [128]],

    [-1, 1, Conv, [256, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 4], 1, Concat, [1]],
    [-1, 3, C3, [256, False]],
    [-1, 1, ResCBAM, [64]],

    [-1, 1, Conv, [256, 3, 2]],
    [[-1, 14], 1, Concat, [1]],
    [-1, 3, C3, [512, False]],
    [-1, 1, ResCBAM, [128]],

    [-1, 1, Conv, [512, 3, 2]],
    [[-1, 10], 1, Concat, [1]],
    [-1, 3, C3, [1024, False]],
    [-1, 1, ResCBAM, [256]],

    [-1, 1, CoordAtt, [256]],
    [[19, 23, 28], 1, Detect, [nc, anchors]],
  ]
""",

    "yolov5n_rescbam4_triplet.yaml": """# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license
nc: 6
depth_multiple: 0.33
width_multiple: 0.25
anchors:
  - [10, 13, 16, 30, 33, 23]
  - [30, 61, 62, 45, 59, 119]
  - [116, 90, 156, 198, 373, 326]

backbone:
  [
    [-1, 1, Conv, [64, 6, 2, 2]],
    [-1, 1, Conv, [128, 3, 2]],
    [-1, 3, C3, [128]],
    [-1, 1, Conv, [256, 3, 2]],
    [-1, 6, C3, [256]],
    [-1, 1, Conv, [512, 3, 2]],
    [-1, 9, C3, [512]],
    [-1, 1, Conv, [1024, 3, 2]],
    [-1, 3, C3, [1024]],
    [-1, 1, SPPF, [1024, 5]],
  ]

head: [
    [-1, 1, Conv, [512, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 6], 1, Concat, [1]],
    [-1, 3, C3, [512, False]],
    [-1, 1, ResCBAM, [128]],

    [-1, 1, Conv, [256, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 4], 1, Concat, [1]],
    [-1, 3, C3, [256, False]],
    [-1, 1, ResCBAM, [64]],

    [-1, 1, Conv, [256, 3, 2]],
    [[-1, 14], 1, Concat, [1]],
    [-1, 3, C3, [512, False]],
    [-1, 1, ResCBAM, [128]],

    [-1, 1, Conv, [512, 3, 2]],
    [[-1, 10], 1, Concat, [1]],
    [-1, 3, C3, [1024, False]],
    [-1, 1, ResCBAM, [256]],

    [-1, 1, TripletAttention, [256]],
    [[19, 23, 28], 1, Detect, [nc, anchors]],
  ]
""",

    "yolov5n_rescbam4_nam.yaml": """# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license
nc: 6
depth_multiple: 0.33
width_multiple: 0.25
anchors:
  - [10, 13, 16, 30, 33, 23]
  - [30, 61, 62, 45, 59, 119]
  - [116, 90, 156, 198, 373, 326]

backbone:
  [
    [-1, 1, Conv, [64, 6, 2, 2]],
    [-1, 1, Conv, [128, 3, 2]],
    [-1, 3, C3, [128]],
    [-1, 1, Conv, [256, 3, 2]],
    [-1, 6, C3, [256]],
    [-1, 1, Conv, [512, 3, 2]],
    [-1, 9, C3, [512]],
    [-1, 1, Conv, [1024, 3, 2]],
    [-1, 3, C3, [1024]],
    [-1, 1, SPPF, [1024, 5]],
  ]

head: [
    [-1, 1, Conv, [512, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 6], 1, Concat, [1]],
    [-1, 3, C3, [512, False]],
    [-1, 1, ResCBAM, [128]],

    [-1, 1, Conv, [256, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 4], 1, Concat, [1]],
    [-1, 3, C3, [256, False]],
    [-1, 1, ResCBAM, [64]],

    [-1, 1, Conv, [256, 3, 2]],
    [[-1, 14], 1, Concat, [1]],
    [-1, 3, C3, [512, False]],
    [-1, 1, ResCBAM, [128]],

    [-1, 1, Conv, [512, 3, 2]],
    [[-1, 10], 1, Concat, [1]],
    [-1, 3, C3, [1024, False]],
    [-1, 1, ResCBAM, [256]],

    [-1, 1, NAM, [256]],
    [[19, 23, 28], 1, Detect, [nc, anchors]],
  ]
""",

    "yolov5n_rescbam4_gcblock.yaml": """# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license
nc: 6
depth_multiple: 0.33
width_multiple: 0.25
anchors:
  - [10, 13, 16, 30, 33, 23]
  - [30, 61, 62, 45, 59, 119]
  - [116, 90, 156, 198, 373, 326]

backbone:
  [
    [-1, 1, Conv, [64, 6, 2, 2]],
    [-1, 1, Conv, [128, 3, 2]],
    [-1, 3, C3, [128]],
    [-1, 1, Conv, [256, 3, 2]],
    [-1, 6, C3, [256]],
    [-1, 1, Conv, [512, 3, 2]],
    [-1, 9, C3, [512]],
    [-1, 1, Conv, [1024, 3, 2]],
    [-1, 3, C3, [1024]],
    [-1, 1, SPPF, [1024, 5]],
  ]

head: [
    [-1, 1, Conv, [512, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 6], 1, Concat, [1]],
    [-1, 3, C3, [512, False]],
    [-1, 1, ResCBAM, [128]],

    [-1, 1, Conv, [256, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 4], 1, Concat, [1]],
    [-1, 3, C3, [256, False]],
    [-1, 1, ResCBAM, [64]],

    [-1, 1, Conv, [256, 3, 2]],
    [[-1, 14], 1, Concat, [1]],
    [-1, 3, C3, [512, False]],
    [-1, 1, ResCBAM, [128]],

    [-1, 1, Conv, [512, 3, 2]],
    [[-1, 10], 1, Concat, [1]],
    [-1, 3, C3, [1024, False]],
    [-1, 1, ResCBAM, [256]],

    [-1, 1, GCBlock, [256]],
    [[19, 23, 28], 1, Detect, [nc, anchors]],
  ]
""",
}

for name, text in model_yamls.items():
    (MODEL_DIR / name).write_text(text, encoding="utf-8")

print("Saved corrected model YAML files:")
for name in model_yamls:
    print(MODEL_DIR / name)


Saved corrected model YAML files:
/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/models_custom/yolov5n_rescbam4_baseline.yaml
/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/models_custom/yolov5n_rescbam4_simam.yaml
/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/models_custom/yolov5n_rescbam4_coordatt.yaml
/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/models_custom/yolov5n_rescbam4_triplet.yaml
/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/models_custom/yolov5n_rescbam4_nam.yaml
/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/models_custom/yolov5n_rescbam4_gcblock.yaml


In [42]:
# parser test for ONE model first
%cd /content/yolov5

one_cfg = str(MODEL_DIR / "yolov5n_rescbam4_simam.yaml")

run_cmd([
    "python",
    "models/yolo.py",
    "--cfg",
    one_cfg
])


/content/yolov5

RUNNING:
python models/yolo.py --cfg /content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/models_custom/yolov5n_rescbam4_simam.yaml

STDERR:

models/yolo: cfg=/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/models_custom/yolov5n_rescbam4_simam.yaml, batch_size=1, device=, profile=False, line_profile=False, test=False
YOLOv5 🚀 v7.0-461-gda75ed9d Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)


                 from  n    params  module                                  arguments                     
  0                -1  1      1760  models.common.Conv                      [3, 16, 6, 2, 2]              
  1                -1  1      4672  models.common.Conv                      [16, 32, 3, 2]                
  2                -1  1      4800  models.common.C3                        [32, 32, 1]                   
  3                -1  1     18560  models.common.Conv             

CompletedProcess(args=['python', 'models/yolo.py', '--cfg', '/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/models_custom/yolov5n_rescbam4_simam.yaml'], returncode=0, stdout='', stderr="\x1b\x1bmodels/yolo: \x1bcfg=/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/models_custom/yolov5n_rescbam4_simam.yaml, batch_size=1, device=, profile=False, line_profile=False, test=False\nYOLOv5 🚀 v7.0-461-gda75ed9d Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)\n\n\n                 from  n    params  module                                  arguments                     \n  0                -1  1      1760  models.common.Conv                      [3, 16, 6, 2, 2]              \n  1                -1  1      4672  models.common.Conv                      [16, 32, 3, 2]                \n  2                -1  1      4800  models.common.C3                        [32, 32, 1]                   \n  3           

In [43]:
# parser test for all models
%cd /content/yolov5

all_cfgs = [
    "yolov5n_rescbam4_baseline.yaml",
    "yolov5n_rescbam4_simam.yaml",
    "yolov5n_rescbam4_coordatt.yaml",
    "yolov5n_rescbam4_triplet.yaml",
    "yolov5n_rescbam4_nam.yaml",
    "yolov5n_rescbam4_gcblock.yaml",
]

for yaml_name in all_cfgs:
    cfg_path = str(MODEL_DIR / yaml_name)
    print(f"\nTesting parser: {yaml_name}")
    run_cmd([
        "python",
        "models/yolo.py",
        "--cfg",
        cfg_path
    ])


/content/yolov5

Testing parser: yolov5n_rescbam4_baseline.yaml

RUNNING:
python models/yolo.py --cfg /content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/models_custom/yolov5n_rescbam4_baseline.yaml

STDERR:

models/yolo: cfg=/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/models_custom/yolov5n_rescbam4_baseline.yaml, batch_size=1, device=, profile=False, line_profile=False, test=False
YOLOv5 🚀 v7.0-461-gda75ed9d Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)


                 from  n    params  module                                  arguments                     
  0                -1  1      1760  models.common.Conv                      [3, 16, 6, 2, 2]              
  1                -1  1      4672  models.common.Conv                      [16, 32, 3, 2]                
  2                -1  1      4800  models.common.C3                        [32, 32, 1]                   
  3          

In [47]:
# one-model debug run first
%cd /content/yolov5

yaml_name = "yolov5n_rescbam4_simam.yaml"
short_name = "simam"

cfg_path = str(MODEL_DIR / yaml_name)
run_name = f"train_{short_name}"

run_cmd([
    "python",
    "train.py",
    "--img", "640",
    "--batch", "16",
    "--epochs", "100",
    "--data", str(data_yaml_path),
    "--cfg", cfg_path,
    "--weights", "",
    "--hyp", str(hyp_yaml_path),
    "--optimizer", "SGD",
    "--cos-lr",
    "--project", str(RUNS_DIR),
    "--name", run_name,
    "--exist-ok",
    "--workers", "2"
])

train_dir = RUNS_DIR / run_name
expected = [
    train_dir / "results.csv",
    train_dir / "weights" / "last.pt",
    train_dir / "weights" / "best.pt",
]
print("\nChecking training artifacts:")
for p in expected:
    print(f"{p} -> {p.exists()}")
    if not p.exists():
        raise FileNotFoundError(f"Missing expected file: {p}")

best_pt = train_dir / "weights" / "best.pt"
run_cmd([
    "python",
    "val.py",
    "--weights", str(best_pt),
    "--data", str(data_yaml_path),
    "--img", "640",
    "--task", "test",
    "--batch", "16",
    "--project", str(RUNS_DIR),
    "--name", f"test_{short_name}",
    "--exist-ok",
    "--save-txt",
    "--save-json"
])


Streaming output truncated to the last 5000 lines.
      98/99      2.97G    0.04285     0.0172    0.00369         53        640:  56%|█████▌    | 500/898 [00:28<00:21, 18.41it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):

      98/99      2.97G    0.04286    0.01721   0.003715         45        640:  56%|█████▌    | 500/898 [00:28<00:21, 18.41it/s]
      98/99      2.97G    0.04286    0.01721   0.003715         45        640:  56%|█████▌    | 503/898 [00:28<00:19, 20.04it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):

      98/99      2.97G    0.04286    0.01722   0.003727         58        640:  56%|█████▌    | 503/898 [00:28<00:19, 20.04it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cu

CompletedProcess(args=['python', 'val.py', '--weights', '/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/runs/train_simam/weights/best.pt', '--data', '/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/data_custom/train_voc_test_dawn.yaml', '--img', '640', '--task', 'test', '--batch', '16', '--project', '/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/runs', '--name', 'test_simam', '--exist-ok', '--save-txt', '--save-json'], returncode=0, stdout='loading annotations into memory...\n', stderr="\x1b\x1bval: \x1bdata=/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/data_custom/train_voc_test_dawn.yaml, weights=['/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/runs/train_simam/weights/best.pt'], batch_size=16, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=test, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=True, save_hybrid=False, sav

In [49]:
# Cell 15 — REPLACE ENTIRE CELL WITH THIS
# train remaining models in a clearer order:
# coordatt -> triplet -> nam -> gcblock -> baseline
# simam is skipped because you already ran it

%cd /content/yolov5

import pandas as pd
from pathlib import Path

model_run_names = [
    ("yolov5n_rescbam4_coordatt.yaml", "coordatt"),
    ("yolov5n_rescbam4_triplet.yaml", "triplet"),
    ("yolov5n_rescbam4_nam.yaml", "nam"),
    ("yolov5n_rescbam4_gcblock.yaml", "gcblock"),
    ("yolov5n_rescbam4_baseline.yaml", "baseline"),
]

all_summaries = []

for idx, (yaml_name, short_name) in enumerate(model_run_names, start=1):
    cfg_path = str(MODEL_DIR / yaml_name)
    run_name = f"train_{short_name}"
    test_name = f"test_{short_name}"

    print("\n" + "#" * 120)
    print(f"MODEL {idx}/{len(model_run_names)}: {short_name.upper()}")
    print(f"YAML: {cfg_path}")
    print("#" * 120)

    # -----------------------------
    # TRAIN
    # -----------------------------
    train_output = run_cmd([
        "python",
        "train.py",
        "--img", "640",
        "--batch", "16",
        "--epochs", "100",
        "--data", str(data_yaml_path),
        "--cfg", cfg_path,
        "--weights", "",
        "--hyp", str(hyp_yaml_path),
        "--optimizer", "SGD",
        "--cos-lr",
        "--project", str(RUNS_DIR),
        "--name", run_name,
        "--exist-ok",
        "--workers", "2"
    ])

    train_dir = RUNS_DIR / run_name
    results_csv = train_dir / "results.csv"
    last_pt = train_dir / "weights" / "last.pt"
    best_pt = train_dir / "weights" / "best.pt"

    print("\n" + "-" * 120)
    print(f"TRAINING FINISHED: {short_name}")
    print("Artifacts:")
    print("results.csv:", results_csv.exists(), results_csv)
    print("last.pt    :", last_pt.exists(), last_pt)
    print("best.pt    :", best_pt.exists(), best_pt)

    if not results_csv.exists():
        raise FileNotFoundError(f"Missing results.csv for {short_name}")
    if not best_pt.exists():
        raise FileNotFoundError(f"Missing best.pt for {short_name}")

    # Best validation row from results.csv
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]

    # Try to detect best epoch by max val mAP50
    val_map50_col = None
    val_map5095_col = None
    precision_col = None
    recall_col = None

    for c in df.columns:
        lc = c.lower()
        if "metrics/map_0.5" in lc or "metrics/map50" in lc:
            if "0.5:0.95" not in lc and "0.5_0.95" not in lc:
                val_map50_col = c
        if "metrics/mAP_0.5:0.95".lower() in lc or "metrics/map_0.5:0.95" in lc or "metrics/map50-95" in lc:
            val_map5095_col = c
        if "metrics/precision" in lc:
            precision_col = c
        if "metrics/recall" in lc:
            recall_col = c

    # Fallback names often used by YOLOv5 CSV
    if val_map50_col is None:
        for c in df.columns:
            if c.strip() == "metrics/mAP_0.5":
                val_map50_col = c
    if val_map5095_col is None:
        for c in df.columns:
            if c.strip() == "metrics/mAP_0.5:0.95":
                val_map5095_col = c
    if precision_col is None:
        for c in df.columns:
            if c.strip() == "metrics/precision":
                precision_col = c
    if recall_col is None:
        for c in df.columns:
            if c.strip() == "metrics/recall":
                recall_col = c

    best_idx = df[val_map50_col].astype(float).idxmax()
    best_row = df.loc[best_idx]

    print("\nBEST VALIDATION SUMMARY")
    print(f"Best epoch   : {int(best_row['epoch']) if 'epoch' in df.columns else best_idx}")
    print(f"Precision    : {float(best_row[precision_col]):.4f}")
    print(f"Recall       : {float(best_row[recall_col]):.4f}")
    print(f"mAP50        : {float(best_row[val_map50_col]):.4f}")
    print(f"mAP50-95     : {float(best_row[val_map5095_col]):.4f}")

    # -----------------------------
    # TEST
    # -----------------------------
    test_output = run_cmd([
        "python",
        "val.py",
        "--weights", str(best_pt),
        "--data", str(data_yaml_path),
        "--img", "640",
        "--task", "test",
        "--batch", "16",
        "--project", str(RUNS_DIR),
        "--name", test_name,
        "--exist-ok",
        "--save-txt",
        "--save-json"
    ])

    parsed = extract_test_all_metrics(test_output)

    print("\nTEST SUMMARY")
    if parsed is not None:
        print(f"Images       : {parsed['images']}")
        print(f"Instances    : {parsed['instances']}")
        print(f"Precision    : {parsed['P']:.4f}")
        print(f"Recall       : {parsed['R']:.4f}")
        print(f"mAP50        : {parsed['mAP50']:.4f}")
        print(f"mAP50-95     : {parsed['mAP50-95']:.4f}")
    else:
        print("Could not auto-parse the final test 'all' row from val.py output.")

    all_summaries.append({
        "model": short_name,
        "val_precision": float(best_row[precision_col]),
        "val_recall": float(best_row[recall_col]),
        "val_mAP50": float(best_row[val_map50_col]),
        "val_mAP50-95": float(best_row[val_map5095_col]),
        "test_precision": None if parsed is None else parsed["P"],
        "test_recall": None if parsed is None else parsed["R"],
        "test_mAP50": None if parsed is None else parsed["mAP50"],
        "test_mAP50-95": None if parsed is None else parsed["mAP50-95"],
    })

    print("\nDONE WITH:", short_name)
    print("=" * 120)

# -----------------------------
# FINAL COMPARISON TABLE
# -----------------------------
summary_df = pd.DataFrame(all_summaries)
summary_path = RUNS_DIR / "summary_remaining_models.csv"
summary_df.to_csv(summary_path, index=False)

print("\n" + "#" * 120)
print("FINAL SUMMARY TABLE")
print("#" * 120)
display(summary_df)
print("\nSaved summary CSV to:", summary_path)


Streaming output truncated to the last 5000 lines.
       7/99      2.93G     0.0634    0.02216    0.01988         50        640:  96%|█████████▌| 858/898 [00:46<00:02, 18.92it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):

       7/99      2.93G     0.0634    0.02215    0.01988         37        640:  96%|█████████▌| 858/898 [00:46<00:02, 18.92it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):

       7/99      2.93G     0.0634    0.02215    0.01987         40        640:  96%|█████████▌| 858/898 [00:46<00:02, 18.92it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast

KeyboardInterrupt: 

In [ ]:
# test all models
%cd /content/yolov5

for _, short_name in model_run_names:
    weights_path = RUNS_DIR / f"train_{short_name}" / "weights" / "best.pt"
    test_name = f"test_{short_name}"

    if not weights_path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {weights_path}")

    run_cmd([
        "python",
        "val.py",
        "--weights", str(weights_path),
        "--data", str(data_yaml_path),
        "--img", "640",
        "--task", "test",
        "--batch", "16",
        "--project", str(RUNS_DIR),
        "--name", test_name,
        "--exist-ok",
        "--save-txt",
        "--save-json"
    ])


In [ ]:
# summarize output paths
from pathlib import Path

print("Training and test output folders:\n")
for _, short_name in model_run_names:
    train_dir = RUNS_DIR / f"train_{short_name}"
    test_dir = RUNS_DIR / f"test_{short_name}"
    best_pt = train_dir / "weights" / "best.pt"

    print(f"{short_name:10s}")
    print(f"  train dir : {train_dir}")
    print(f"  best.pt   : {best_pt}")
    print(f"  test dir  : {test_dir}")
    print("-" * 100)


In [ ]:
# resume one interrupted run
%cd /content/yolov5

short_name = "simam"  # change this
last_ckpt = RUNS_DIR / f"train_{short_name}" / "weights" / "last.pt"

if not last_ckpt.exists():
    raise FileNotFoundError(last_ckpt)

run_cmd([
    "python",
    "train.py",
    "--resume", str(last_ckpt)
])


In [ ]:
# train and test just one chosen model
%cd /content/yolov5

yaml_name = "yolov5n_rescbam4_gcblock.yaml"  # change this
short_name = "gcblock"                       # change this

cfg_path = str(MODEL_DIR / yaml_name)
run_name = f"train_{short_name}"

run_cmd([
    "python",
    "models/yolo.py",
    "--cfg",
    cfg_path
])

run_cmd([
    "python",
    "train.py",
    "--img", "640",
    "--batch", "16",
    "--epochs", "100",
    "--data", str(data_yaml_path),
    "--cfg", cfg_path,
    "--weights", "",
    "--hyp", str(hyp_yaml_path),
    "--optimizer", "SGD",
    "--cos-lr",
    "--project", str(RUNS_DIR),
    "--name", run_name,
    "--exist-ok",
    "--workers", "2"
])

train_dir = RUNS_DIR / run_name
best_pt = train_dir / "weights" / "best.pt"
if not best_pt.exists():
    raise FileNotFoundError(best_pt)

run_cmd([
    "python",
    "val.py",
    "--weights", str(best_pt),
    "--data", str(data_yaml_path),
    "--img", "640",
    "--task", "test",
    "--batch", "16",
    "--project", str(RUNS_DIR),
    "--name", f"test_{short_name}",
    "--exist-ok",
    "--save-txt",
    "--save-json"
])


In [54]:
# COMPREHENSIVE RESULTS CELL FOR FINISHED NON-BASELINE MODELS
%cd /content/yolov5

from pathlib import Path
import pandas as pd

finished_models = ["simam", "coordatt", "triplet", "nam", "gcblock"]

def get_best_val_metrics(results_csv_path):
    df = pd.read_csv(results_csv_path)
    df.columns = [c.strip() for c in df.columns]

    val_map50_col = None
    val_map5095_col = None
    precision_col = None
    recall_col = None

    for c in df.columns:
        lc = c.lower()
        if ("metrics/map_0.5" in lc or "metrics/map50" in lc) and ("0.5:0.95" not in lc and "0.5_0.95" not in lc):
            val_map50_col = c
        if "metrics/map_0.5:0.95" in lc or "metrics/map50-95" in lc:
            val_map5095_col = c
        if "metrics/precision" in lc:
            precision_col = c
        if "metrics/recall" in lc:
            recall_col = c

    if val_map50_col is None and "metrics/mAP_0.5" in df.columns:
        val_map50_col = "metrics/mAP_0.5"
    if val_map5095_col is None and "metrics/mAP_0.5:0.95" in df.columns:
        val_map5095_col = "metrics/mAP_0.5:0.95"
    if precision_col is None and "metrics/precision" in df.columns:
        precision_col = "metrics/precision"
    if recall_col is None and "metrics/recall" in df.columns:
        recall_col = "metrics/recall"

    best_idx = df[val_map50_col].astype(float).idxmax()
    best_row = df.loc[best_idx]

    return {
        "best_epoch": int(best_row["epoch"]) if "epoch" in df.columns else int(best_idx),
        "val_precision": float(best_row[precision_col]),
        "val_recall": float(best_row[recall_col]),
        "val_mAP50": float(best_row[val_map50_col]),
        "val_mAP50-95": float(best_row[val_map5095_col]),
    }

summary_rows = []

for model_name in finished_models:
    print("\n" + "=" * 120)
    print(f"PROCESSING: {model_name.upper()}")

    train_dir = RUNS_DIR / f"train_{model_name}"
    best_pt = train_dir / "weights" / "best.pt"
    results_csv = train_dir / "results.csv"

    if not train_dir.exists():
        print(f"Skipping {model_name}: missing train dir {train_dir}")
        continue
    if not best_pt.exists():
        print(f"Skipping {model_name}: missing best.pt")
        continue
    if not results_csv.exists():
        print(f"Skipping {model_name}: missing results.csv")
        continue

    val_metrics = get_best_val_metrics(results_csv)

    print("Best validation metrics:")
    print(val_metrics)

    test_name = f"test_{model_name}_rerun"

    test_output = run_cmd([
        "python",
        "val.py",
        "--weights", str(best_pt),
        "--data", str(data_yaml_path),
        "--img", "640",
        "--task", "test",
        "--batch", "16",
        "--project", str(RUNS_DIR),
        "--name", test_name,
        "--exist-ok",
        "--save-txt",
        "--save-json"
    ])

    parsed = extract_test_all_metrics(test_output)

    if parsed is None:
        print(f"Could not parse final test row for {model_name}")
        parsed = {
            "images": None,
            "instances": None,
            "P": None,
            "R": None,
            "mAP50": None,
            "mAP50-95": None,
        }

    print("Test metrics:")
    print(parsed)

    summary_rows.append({
        "model": model_name,
        "best_epoch": val_metrics["best_epoch"],
        "val_precision": val_metrics["val_precision"],
        "val_recall": val_metrics["val_recall"],
        "val_mAP50": val_metrics["val_mAP50"],
        "val_mAP50-95": val_metrics["val_mAP50-95"],
        "test_precision": parsed["P"],
        "test_recall": parsed["R"],
        "test_mAP50": parsed["mAP50"],
        "test_mAP50-95": parsed["mAP50-95"],
    })

summary_df = pd.DataFrame(summary_rows)

if not summary_df.empty:
    summary_df = summary_df.sort_values(by="test_mAP50", ascending=False, na_position="last").reset_index(drop=True)

summary_path = RUNS_DIR / "summary_finished_models_no_baseline.csv"
summary_df.to_csv(summary_path, index=False)

print("\n" + "#" * 120)
print("FINAL COMPARISON TABLE")
print("#" * 120)
display(summary_df)
print("\nSaved to:", summary_path)


/content/yolov5

PROCESSING: SIMAM
Best validation metrics:
{'best_epoch': 94, 'val_precision': 0.73474, 'val_recall': 0.60588, 'val_mAP50': 0.66102, 'val_mAP50-95': 0.43628}

RUNNING:
python val.py --weights /content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/runs/train_simam/weights/best.pt --data /content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/data_custom/train_voc_test_dawn.yaml --img 640 --task test --batch 16 --project /content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/runs --name test_simam_rerun --exist-ok --save-txt --save-json
val: data=/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/data_custom/train_voc_test_dawn.yaml, weights=['/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/runs/train_simam/weights/best.pt'], batch_size=16, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=test, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt

,model,best_epoch,val_precision,val_recall,val_mAP50,val_mAP50-95,test_precision,test_recall,test_mAP50,test_mAP50-95
0,coordatt,99,0.72798,0.59328,0.66310,0.43678,0.437,0.252,0.260,0.145
1,simam,94,0.73474,0.60588,0.66102,0.43628,0.505,0.230,0.257,0.146
2,nam,98,0.72249,0.60492,0.66193,0.44193,0.395,0.243,0.251,0.139
3,triplet,96,0.72622,0.59916,0.66373,0.43900,0.362,0.247,0.238,0.133
4,gcblock,91,0.72614,0.61729,0.66805,0.44414,0.356,0.231,0.231,0.132



Saved to: /content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/runs/summary_finished_models_no_baseline.csv


In [61]:
# TRAIN + TEST BASELINE ONLY
%cd /content/yolov5

yaml_name = "yolov5n_rescbam4_baseline.yaml"
short_name = "baseline"

cfg_path = str(MODEL_DIR / yaml_name)
run_name = f"train_{short_name}"
test_name = f"test_{short_name}"

print("\n" + "#" * 120)
print("TRAINING BASELINE: YOLOv5n + ResCBAM x4")
print(f"YAML: {cfg_path}")
print("#" * 120)

# -----------------------------
# TRAIN
# -----------------------------
train_output = run_cmd([
    "python",
    "train.py",
    "--img", "640",
    "--batch", "16",
    "--epochs", "100",
    "--data", str(data_yaml_path),
    "--cfg", cfg_path,
    "--weights", "",
    "--hyp", str(hyp_yaml_path),
    "--optimizer", "SGD",
    "--cos-lr",
    "--project", str(RUNS_DIR),
    "--name", run_name,
    "--exist-ok",
    "--workers", "2"
])

train_dir = RUNS_DIR / run_name
results_csv = train_dir / "results.csv"
last_pt = train_dir / "weights" / "last.pt"
best_pt = train_dir / "weights" / "best.pt"

print("\n" + "-" * 120)
print("TRAINING FINISHED: baseline")
print("Artifacts:")
print("results.csv:", results_csv.exists(), results_csv)
print("last.pt    :", last_pt.exists(), last_pt)
print("best.pt    :", best_pt.exists(), best_pt)

if not results_csv.exists():
    raise FileNotFoundError(f"Missing results.csv: {results_csv}")
if not best_pt.exists():
    raise FileNotFoundError(f"Missing best.pt: {best_pt}")

# -----------------------------
# BEST VALIDATION SUMMARY
# -----------------------------
df = pd.read_csv(results_csv)
df.columns = [c.strip() for c in df.columns]

val_map50_col = None
val_map5095_col = None
precision_col = None
recall_col = None

for c in df.columns:
    lc = c.lower()
    if ("metrics/map_0.5" in lc or "metrics/map50" in lc) and ("0.5:0.95" not in lc and "0.5_0.95" not in lc):
        val_map50_col = c
    if "metrics/map_0.5:0.95" in lc or "metrics/map50-95" in lc:
        val_map5095_col = c
    if "metrics/precision" in lc:
        precision_col = c
    if "metrics/recall" in lc:
        recall_col = c

if val_map50_col is None and "metrics/mAP_0.5" in df.columns:
    val_map50_col = "metrics/mAP_0.5"
if val_map5095_col is None and "metrics/mAP_0.5:0.95" in df.columns:
    val_map5095_col = "metrics/mAP_0.5:0.95"
if precision_col is None and "metrics/precision" in df.columns:
    precision_col = "metrics/precision"
if recall_col is None and "metrics/recall" in df.columns:
    recall_col = "metrics/recall"

best_idx = df[val_map50_col].astype(float).idxmax()
best_row = df.loc[best_idx]

print("\nBEST VALIDATION SUMMARY")
print(f"Best epoch   : {int(best_row['epoch']) if 'epoch' in df.columns else best_idx}")
print(f"Precision    : {float(best_row[precision_col]):.4f}")
print(f"Recall       : {float(best_row[recall_col]):.4f}")
print(f"mAP50        : {float(best_row[val_map50_col]):.4f}")
print(f"mAP50-95     : {float(best_row[val_map5095_col]):.4f}")

# -----------------------------
# TEST
# -----------------------------
test_output = run_cmd([
    "python",
    "val.py",
    "--weights", str(best_pt),
    "--data", str(data_yaml_path),
    "--img", "640",
    "--task", "test",
    "--batch", "16",
    "--project", str(RUNS_DIR),
    "--name", test_name,
    "--exist-ok",
    "--save-txt",
    "--save-json"
])

parsed = extract_test_all_metrics(test_output)

print("\nTEST SUMMARY")
if parsed is not None:
    print(f"Images       : {parsed['images']}")
    print(f"Instances    : {parsed['instances']}")
    print(f"Precision    : {parsed['P']:.4f}")
    print(f"Recall       : {parsed['R']:.4f}")
    print(f"mAP50        : {parsed['mAP50']:.4f}")
    print(f"mAP50-95     : {parsed['mAP50-95']:.4f}")
else:
    print("Could not auto-parse the final test 'all' row from val.py output.")

print("\nDONE WITH: baseline")
print("train dir:", train_dir)
print("test dir :", RUNS_DIR / test_name)

Streaming output truncated to the last 5000 lines.
      98/99      2.93G    0.04308    0.01712   0.003758         26        640:  47%|████▋     | 421/898 [00:23<00:33, 14.29it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):

      98/99      2.93G    0.04306    0.01711   0.003752         30        640:  47%|████▋     | 421/898 [00:23<00:33, 14.29it/s]
      98/99      2.93G    0.04306    0.01711   0.003752         30        640:  47%|████▋     | 424/898 [00:23<00:30, 15.58it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):

      98/99      2.93G    0.04305    0.01711   0.003752         39        640:  47%|████▋     | 424/898 [00:23<00:30, 15.58it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cu

In [62]:
# TEST BASELINE ONLY
%cd /content/yolov5

short_name = "baseline"
best_pt = RUNS_DIR / f"train_{short_name}" / "weights" / "best.pt"
test_name = f"test_{short_name}"

if not best_pt.exists():
    raise FileNotFoundError(f"Missing checkpoint: {best_pt}")

test_output = run_cmd([
    "python",
    "val.py",
    "--weights", str(best_pt),
    "--data", str(data_yaml_path),
    "--img", "640",
    "--task", "test",
    "--batch", "16",
    "--project", str(RUNS_DIR),
    "--name", test_name,
    "--exist-ok",
    "--save-txt",
    "--save-json"
])

parsed = extract_test_all_metrics(test_output)

print("\n" + "=" * 100)
print("BASELINE TEST SUMMARY")
print("=" * 100)

if parsed is not None:
    print(f"Images       : {parsed['images']}")
    print(f"Instances    : {parsed['instances']}")
    print(f"Precision    : {parsed['P']:.4f}")
    print(f"Recall       : {parsed['R']:.4f}")
    print(f"mAP50        : {parsed['mAP50']:.4f}")
    print(f"mAP50-95     : {parsed['mAP50-95']:.4f}")
else:
    print("Could not auto-parse the final test 'all' row from val.py output.")

print("\nSaved to:", RUNS_DIR / test_name)

/content/yolov5

RUNNING:
python val.py --weights /content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/runs/train_baseline/weights/best.pt --data /content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/data_custom/train_voc_test_dawn.yaml --img 640 --task test --batch 16 --project /content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/runs --name test_baseline --exist-ok --save-txt --save-json
val: data=/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/data_custom/train_voc_test_dawn.yaml, weights=['/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/runs/train_baseline/weights/best.pt'], batch_size=16, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=test, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=True, save_hybrid=False, save_conf=False, save_json=True, project=/content/drive/MyDrive/Lab thầy Vinh/yolov5_rescbam4_attention_study/runs, name=test_ba